# E007 trên V13 — Google Colab
Dùng tài khoản Google AI Pro đã được owner cho phép. Upload `e007-colab-bundle.zip` vào **My Drive/E007/** trước khi kết nối GPU. Giữ thư mục riêng tư.

Chọn **Runtime → Change runtime type → GPU mạnh nhất đang được cấp**, bật High RAM nếu có. Không mua thêm credits. Bấm **Run all**, cấp quyền mount Drive đúng tài khoản. Notebook tự chạy smoke → 5 đối chứng → xác nhận champion và R0, tối đa 9 research run.

Bộ đếm 8 giờ bắt đầu khi chạy cell; bao gồm cài dependencies và thời gian chờ từ đó. Kết nối GPU rồi chạy ngay để tránh tiêu CU trước bộ đếm. GPU/CU thực tế phụ thuộc tài khoản và khả dụng, không được bảo đảm chỉ từ tên gói.

Kết quả ở **My Drive/E007/E007-V13-COLAB-20261024/**. Runtime tự ngắt khi xong hoặc lỗi; checkpoint epoch đã đồng bộ nằm trong `recovery/`. Nếu bị ngắt: giữ nguyên folder và log, không chạy lại toàn notebook để tạo run trùng. Không mở test hoặc tự đổi batch/precision/recipe khi OOM.


In [ ]:
import hashlib, json, os, subprocess, sys, threading, time, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, runtime

EXPECTED_SHA256 = '254596ddc336e3516cb39a7b7cf82ec9e1a4c6172aa650686b2df57574174c5c'
started_at = time.time()
watchdog = threading.Timer(8 * 3600, runtime.unassign)
watchdog.daemon = True
watchdog.start()
storage = None
try:
    if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
        raise RuntimeError('This approved environment requires Python 3.11 or 3.12')
    subprocess.run(['nvidia-smi'], check=True)
    drive.mount('/content/drive')
    storage = Path('/content/drive/MyDrive/E007')
    archive_path = storage / 'e007-colab-bundle.zip'
    digest = hashlib.sha256()
    with archive_path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    if digest.hexdigest() != EXPECTED_SHA256:
        raise RuntimeError('Bundle checksum mismatch; stop without executing its code')
    root = Path('/content/e007')
    if root.exists() or (storage / 'E007-V13-COLAB-20261024').exists():
        raise RuntimeError('Existing campaign: preserve it and use explicit recovery')
    root.mkdir()
    with zipfile.ZipFile(archive_path) as archive:
        manifest = json.loads(archive.read('bundle-manifest.json'))
        names = archive.namelist()
        if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'bundle-manifest.json'}:
            raise RuntimeError('Unexpected bundle inventory')
        for name, expected in manifest['files'].items():
            path = PurePosixPath(name)
            if path.is_absolute() or '..' in path.parts or '\\' in name or ':' in name:
                raise RuntimeError('Unsafe bundle path')
            payload = archive.read(name)
            if hashlib.sha256(payload).hexdigest() != expected:
                raise RuntimeError('Bundle member checksum mismatch: ' + name)
            destination = root / name
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(payload)
        (root / 'bundle-manifest.json').write_text(json.dumps(manifest, indent=2))
    setup_log = storage / ('setup-' + str(int(started_at)) + '.log')
    venv = Path('/content/e007-venv')
    python = venv / 'bin/python'
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    os.environ['XFORMERS_DISABLED'] = '1'
    with setup_log.open('w') as log:
        commands = [
            [sys.executable, '-m', 'venv', str(venv)],
            [str(python), '-m', 'pip', 'install', '-r', 'requirements/classifier-colab.txt'],
            [str(python), '-m', 'pip', 'install', '--no-deps', '-e', '.'],
        ]
        for command in commands:
            subprocess.run(command, cwd=root, stdout=log, stderr=subprocess.STDOUT, check=True)
    freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], cwd=root, text=True)
    (storage / ('environment-' + str(int(started_at)) + '.txt')).write_text(freeze)
    print('Setup complete. Logs and epoch recovery are saved to private Drive.', flush=True)
    subprocess.run([str(python), '-m', 'ai_exam_monitoring.training.representation_campaign',
                    '--workspace', str(root), '--storage', str(storage),
                    '--design', str(root / 'configs/experiments/E007-V13-20261024/design.json'),
                    '--gpu-started-at', str(started_at)], cwd=root, check=True)
    print('Campaign finished. Review campaign.json and metrics in Drive; no model promotion implied.')
except BaseException as error:
    if storage is not None and storage.is_dir():
        (storage / ('launcher-error-' + str(int(started_at)) + '.txt')).write_text(repr(error))
    raise
finally:
    watchdog.cancel()
    runtime.unassign()
